In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import sys

project_root = Path().cwd().parents[0]
sys.path.insert(0, str(project_root))

from scripts.explanatory_data_analysis import (
    fetch_dataset_and_turn_into_dataframe_and_convert_column_names_to_lowercase,
    dataset_quality_inspection,
)
from scripts.data_preprocessing import (
    data_clues_and_target,
    data_train_test_split,
    identify_feature_groups,
    create_numerical_imputer,
    create_categorical_imputer,
    create_ordinal_encoder,
    create_binary_encoder,
    create_nominal_encoder,
    create_social_media_screen_time_transformer,
    create_numerical_scaler,
    create_numerical_pipeline,
    create_ordinal_pipeline,
    create_nominal_pipeline,
    create_binary_pipeline,
    create_preprocessor,
)

from scripts.model_training import (
    create_model_pipeline,
    evaluate_model_with_cross_validation
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

raw_data_path = project_root / "data" / "raw" / "student_mental_health.csv"

raw_df = fetch_dataset_and_turn_into_dataframe_and_convert_column_names_to_lowercase(
    raw_data_path
)

clues_and_target = data_clues_and_target(raw_df)

# DataFrame Model Clues
X = clues_and_target["X"]
# DataFrame Model Target
y = clues_and_target["y"]


display(X.columns)
display(y)

Index(['age', 'gender', 'year_of_study', 'sleep_hours', 'study_hours_per_week',
       'gpa', 'social_media_hours', 'screen_time_hours', 'physical_activity',
       'financial_stress', 'family_support', 'attendance_rate',
       'part_time_job', 'diet_quality', 'previous_mental_health_history'],
      dtype='str')

0         High
1       Medium
2       Medium
3         High
4       Medium
         ...  
1995    Medium
1996      High
1997      High
1998       Low
1999    Medium
Name: mental_health_risk, Length: 2000, dtype: str

In [2]:
# Data Train Test Split
X_train, X_test, y_train, y_test = data_train_test_split(raw_df)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape[0])
print(y_test.shape[0])

print("=" * 30)

# Make sure the traget distribution is balanced
print("TRAIN TARGET DISTRIBUTION")
print(y_train.value_counts(normalize=True) * 100)
print("=" * 30)
print("TEST TARGET DISTRIBUTION")
print(y_test.value_counts(normalize=True) * 100)

(1600, 15)
(400, 15)
1600
400
TRAIN TARGET DISTRIBUTION
mental_health_risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64
TEST TARGET DISTRIBUTION
mental_health_risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64


In [3]:
# Identify data feature groups
feature_groups = identify_feature_groups(X_train)
numerical_group = feature_groups["numerical_group"]
ordinal_categorical_group = feature_groups["ordinal_categorical_group"]
nominal_categorical_group = feature_groups["nominal_categorical_group"]
binary_categorical_group = feature_groups["binary_categorical_group"]

print(f"NUMERICAL GROUP: {numerical_group}")
print("=" * 30)
print(f"ORDINAL CATEGORICAL GROUP: {ordinal_categorical_group}")
print("=" * 30)
print(f"NOMINAL CATEGORICAL GROUP: {nominal_categorical_group}")
print("=" * 30)
print(f"BINARY CATEGOIRICAL GROUP: {binary_categorical_group}")

NUMERICAL GROUP: ['age', 'year_of_study', 'sleep_hours', 'study_hours_per_week', 'gpa', 'social_media_hours', 'screen_time_hours', 'financial_stress', 'family_support', 'attendance_rate']
ORDINAL CATEGORICAL GROUP: ['physical_activity', 'diet_quality']
NOMINAL CATEGORICAL GROUP: ['gender']
BINARY CATEGOIRICAL GROUP: ['part_time_job', 'previous_mental_health_history']


In [4]:
# Initialize numerical pipeline
numerical_pipeline = create_numerical_pipeline(
    imputer=create_numerical_imputer(),
    transformer=create_social_media_screen_time_transformer(),
    scaler=create_numerical_scaler(),
)

# Initialize ordinal pipeline
ordinal_pipeline = create_ordinal_pipeline(
    imputer=create_categorical_imputer(), encoder=create_ordinal_encoder()
)

# Initialize nominal pipeline
nominal_pipeline = create_nominal_pipeline(
    imputer=create_categorical_imputer(), encoder=create_nominal_encoder()
)

# Initialize binary pipeline
binary_pipeline = create_binary_pipeline(
    imputer=create_categorical_imputer(), encoder=create_binary_encoder()
)

# Initialize preprocessor (ColumnTransformer)
preprocessor = create_preprocessor(
    numerical_pipeline=numerical_pipeline,
    ordinal_pipeline=ordinal_pipeline,
    nominal_pipeline=nominal_pipeline,
    binary_pipeline=binary_pipeline,
    numerical_group=numerical_group,
    ordinal_categorical_group=ordinal_categorical_group,
    nominal_categorical_group=nominal_categorical_group,
    binary_categorical_group=binary_categorical_group,
)

In [5]:
## Initialize models pipelines

# Initialize LogisticRegression pipeline 
logistic_regression_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_regression_pipeline = create_model_pipeline(preprocessor=preprocessor, classifier=logistic_regression_model)

# Initialize RandromForestClassifier pipeline
random_forest_classifier_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight=None,
    random_state=42,
    n_jobs=-1
)
random_forest_classifier_pipeline = create_model_pipeline(preprocessor=preprocessor, classifier=random_forest_classifier_model)

# Initialize Support Vector Classifier pipeline
support_vector_model = SVC(
    C=1.0,
    kernel="rbf",
    gamma="scale",
    class_weight=None
)
support_vector_model_pipeline = create_model_pipeline(preprocessor=preprocessor, classifier=support_vector_model)

In [12]:
# Testing models using cross validation

## LOGISTIC REGRESSION
lr_cv_results = evaluate_model_with_cross_validation(
    pipeline=logistic_regression_pipeline, X_train=X_train, y_train=y_train
)
lr_cv_results_df = pd.DataFrame(lr_cv_results)
print("LOGISTIC REGRESSION RESULTS")
display(lr_cv_results_df)

## RANDOM FOREST
rf_cv_results = evaluate_model_with_cross_validation(
    pipeline=random_forest_classifier_pipeline, X_train=X_train, y_train=y_train
)
rf_cv_results_df = pd.DataFrame(rf_cv_results)
print("RANDOM FORREST RESULTS")
display(rf_cv_results_df)

## SUPPORT VECOTR CLASSIFIER
svc_cv_results = evaluate_model_with_cross_validation(
    pipeline=support_vector_model_pipeline, X_train=X_train, y_train=y_train
)
svc_cv_results_df = pd.DataFrame(svc_cv_results)
print("SUPPORT VECTOR CLASSIFIER RESULTS")
display(svc_cv_results_df)

LOGISTIC REGRESSION RESULTS


,fit_time,score_time,test_accuracy,train_accuracy,test_precision_macro,train_precision_macro,test_recall_macro,train_recall_macro,test_f1_macro,train_f1_macro
0,0.045322,0.016631,0.750000,0.759375,0.745571,0.757934,0.749418,0.758794,0.746957,0.758340
1,0.037625,0.018194,0.731250,0.770312,0.737886,0.770327,0.731268,0.769665,0.733819,0.769979
2,0.039178,0.018945,0.734375,0.767969,0.729607,0.769294,0.733633,0.767589,0.731095,0.768329
3,0.047288,0.018492,0.753125,0.764062,0.753688,0.765405,0.751846,0.763609,0.751681,0.764425
4,0.044773,0.018476,0.787500,0.758594,0.793632,0.759485,0.787346,0.758065,0.789301,0.758724


RANDOM FORREST RESULTS


,fit_time,score_time,test_accuracy,train_accuracy,test_precision_macro,train_precision_macro,test_recall_macro,train_recall_macro,test_f1_macro,train_f1_macro
0,0.303584,0.061510,0.709375,0.993750,0.701116,0.993735,0.708362,0.993706,0.702216,0.993718
1,0.374840,0.070953,0.653125,0.994531,0.652677,0.994561,0.653021,0.994496,0.652807,0.994518
2,0.334819,0.070470,0.681250,0.989844,0.679862,0.989837,0.680777,0.989745,0.679538,0.989776
3,0.355574,0.077110,0.693750,0.996875,0.684036,0.996924,0.691878,0.996866,0.684750,0.996891
4,0.356666,0.087601,0.650000,0.994531,0.639384,0.994521,0.648339,0.994478,0.642547,0.994497


SUPPORT VECTOR CLASSIFIER RESULTS


,fit_time,score_time,test_accuracy,train_accuracy,test_precision_macro,train_precision_macro,test_recall_macro,train_recall_macro,test_f1_macro,train_f1_macro
0,0.048191,0.051458,0.743750,0.834375,0.741753,0.839438,0.743303,0.834446,0.742310,0.835729
1,0.062425,0.044749,0.703125,0.839063,0.707878,0.841149,0.702992,0.838831,0.705078,0.839624
2,0.062695,0.041024,0.706250,0.840625,0.714050,0.843704,0.706224,0.840521,0.708676,0.841502
3,0.059199,0.045271,0.740625,0.832812,0.743990,0.835368,0.739673,0.832652,0.741096,0.833508
4,0.066267,0.037237,0.725000,0.821875,0.729788,0.823683,0.724785,0.821581,0.725847,0.822408
